In [1]:
!pip install -q gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.


In [2]:
from gtts import gTTS

print("Text-to-Speech ready ✅")

Text-to-Speech ready ✅


In [3]:
lesson_text = """
Photosynthesis helps green plants make food.

Plants use sunlight, carbon dioxide, and water to make glucose.
Oxygen is released as a byproduct.

Photosynthesis happens inside chloroplasts in plant cells.
"""

print(lesson_text)


Photosynthesis helps green plants make food.

Plants use sunlight, carbon dioxide, and water to make glucose.
Oxygen is released as a byproduct.

Photosynthesis happens inside chloroplasts in plant cells.



In [4]:
tts = gTTS(
    text=lesson_text,
    lang="en",
    slow=False
)

tts.save("photosynthesis_lesson.mp3")

print("Audio generated successfully! 🔊")

Audio generated successfully! 🔊


In [5]:
from IPython.display import Audio, display

display(Audio("photosynthesis_lesson.mp3"))

In [6]:
lesson_text = """
Photosynthesis helps green plants make food.

Plants use sunlight, carbon dioxide, and water to make glucose.
Oxygen is released as a byproduct.

Photosynthesis happens inside chloroplasts in plant cells.
"""

print("Accessible lesson loaded ✅")
print(lesson_text)

Accessible lesson loaded ✅

Photosynthesis helps green plants make food.

Plants use sunlight, carbon dioxide, and water to make glucose.
Oxygen is released as a byproduct.

Photosynthesis happens inside chloroplasts in plant cells.



In [7]:
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.5.0 which is incompatible.


In [8]:
import gradio as gr

print("Gradio imported successfully ✅")

Gradio imported successfully ✅


In [9]:
import gradio as gr
from gtts import gTTS
import re
import html


def create_reading_view(text, font_size, line_spacing, letter_spacing):

    # Escape HTML for safety
    text = html.escape(text)

    # Convert Markdown headings to HTML
    text = re.sub(
        r"^### (.+)$",
        r"<h3>\1</h3>",
        text,
        flags=re.MULTILINE
    )

    text = re.sub(
        r"^## (.+)$",
        r"<h2>\1</h2>",
        text,
        flags=re.MULTILINE
    )

    text = re.sub(
        r"^# (.+)$",
        r"<h1>\1</h1>",
        text,
        flags=re.MULTILINE
    )

    # Convert bold Markdown
    text = re.sub(
        r"\*\*(.+?)\*\*",
        r"<strong>\1</strong>",
        text
    )

    # Convert horizontal lines
    text = re.sub(
        r"^---+$",
        r"<hr>",
        text,
        flags=re.MULTILINE
    )

    # Convert normal lines into paragraphs
    lines = text.split("\n")

    formatted_lines = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        if (
            line.startswith("<h1>")
            or line.startswith("<h2>")
            or line.startswith("<h3>")
            or line.startswith("<hr>")
        ):
            formatted_lines.append(line)

        else:
            formatted_lines.append(
                f"<p>{line}</p>"
            )

    formatted_text = "\n".join(formatted_lines)

    return f"""
    <div style="
        font-size: {font_size}px;
        line-height: {line_spacing};
        letter-spacing: {letter_spacing}px;
        font-family: Verdana, Arial, sans-serif;
        max-width: 850px;
        margin: 20px auto;
        padding: 35px;
        background: #fffdf5;
        color: #222222;
        border-radius: 12px;
        border: 1px solid #dddddd;
    ">

        {formatted_text}

    </div>
    """


def generate_audio(text):

    # Remove Markdown symbols for speech
    clean_text = re.sub(r"[#*_`]", "", text)

    tts = gTTS(
        text=clean_text,
        lang="en",
        slow=False
    )

    audio_file = "lesson_audio.mp3"

    tts.save(audio_file)

    return audio_file


with gr.Blocks(title="Accessible Learning") as demo:

    gr.Markdown("# 📚 Accessible Learning")

    gr.Markdown(
        "## 📖 Dyslexia-Friendly Reading Mode"
    )

    # -----------------------------
    # Lesson Content
    # -----------------------------

    text_input = gr.Textbox(
        value=lesson_text,
        label="Lesson Content",
        lines=12
    )

    # -----------------------------
    # Reading Controls
    # -----------------------------

    font_size = gr.Slider(
        minimum=16,
        maximum=32,
        value=22,
        step=2,
        label="Text Size"
    )

    line_spacing = gr.Slider(
        minimum=1.2,
        maximum=2.5,
        value=1.8,
        step=0.1,
        label="Line Spacing"
    )

    letter_spacing = gr.Slider(
        minimum=0,
        maximum=5,
        value=1,
        step=0.5,
        label="Letter Spacing"
    )

    # -----------------------------
    # Reading Mode
    # -----------------------------

    update_button = gr.Button(
        "📖 Apply Reading Mode"
    )

    output = gr.HTML()

    update_button.click(
        fn=create_reading_view,
        inputs=[
            text_input,
            font_size,
            line_spacing,
            letter_spacing
        ],
        outputs=output
    )

    # -----------------------------
    # Text-to-Speech
    # -----------------------------

    gr.Markdown(
        "## 🔊 Listen to the Lesson"
    )

    listen_button = gr.Button(
        "🔊 Generate Audio"
    )

    audio_output = gr.Audio(
        label="Lesson Audio",
        type="filepath"
    )

    listen_button.click(
        fn=generate_audio,
        inputs=text_input,
        outputs=audio_output
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://382cc0f39e130c3fe9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
import json

with open("universal_lesson.json", "r", encoding="utf-8") as f:
    universal_lesson = json.load(f)

print("Universal Lesson loaded ✅")
print("Title:", universal_lesson["title"])
print("Subject:", universal_lesson["subject"])

Universal Lesson loaded ✅
Title: Photosynthesis: Nature's Solar Panels
Subject: Middle School Science (Biology / Life Science)


In [20]:
lesson_text = ""

for section in universal_lesson["sections"]:
    lesson_text += section["easy_reading_content"] + "\n\n"

print("Real AI lesson text loaded ✅")
print(lesson_text)

Real AI lesson text loaded ✅
# Photosynthesis: Nature's Solar Panels
### A Middle School Science Guide

---

## Introduction

Have you ever wondered how plants get their food?

Animals eat food to get energy. But plants are different. Plants **make their own food**.

Plants use three things to make food:
- **Sunlight**
- **Water**
- **Carbon dioxide** (a gas in the air)

This process is called **photosynthesis**.

Photosynthesis is one of the most important processes on Earth. Without it, most living things could not survive.

In this lesson, you will learn:
- How photosynthesis works
- Why it is important
- How scientists want to use this process to make clean energy

---

## The Photosynthesis Equation

Photosynthesis is a **chemical reaction**. A chemical reaction is when substances combine and change into something new.

In photosynthesis, light energy is turned into **chemical energy** that the plant can use.

Scientists write it like this:

> **Carbon Dioxide + Water + Sunlight →

In [21]:
lesson_text = ""

for section in universal_lesson["sections"]:
    content = section.get("easy_reading_content", "")

    if content:
        lesson_text += content + "\n\n"

print("Real AI lesson text loaded ✅")
print(lesson_text)

Real AI lesson text loaded ✅
# Photosynthesis: Nature's Solar Panels
### A Middle School Science Guide

---

## Introduction

Have you ever wondered how plants get their food?

Animals eat food to get energy. But plants are different. Plants **make their own food**.

Plants use three things to make food:
- **Sunlight**
- **Water**
- **Carbon dioxide** (a gas in the air)

This process is called **photosynthesis**.

Photosynthesis is one of the most important processes on Earth. Without it, most living things could not survive.

In this lesson, you will learn:
- How photosynthesis works
- Why it is important
- How scientists want to use this process to make clean energy

---

## The Photosynthesis Equation

Photosynthesis is a **chemical reaction**. A chemical reaction is when substances combine and change into something new.

In photosynthesis, light energy is turned into **chemical energy** that the plant can use.

Scientists write it like this:

> **Carbon Dioxide + Water + Sunlight →

In [14]:
from gtts import gTTS

test_text = "This is a test of the accessible learning text to speech system."

tts = gTTS(
    text=test_text,
    lang="en",
    slow=False
)

tts.save("test_audio.mp3")

print("TTS audio created successfully ✅")

TTS audio created successfully ✅


In [15]:
from IPython.display import Audio, display

display(Audio("test_audio.mp3"))

In [16]:
!pip install -q "click<8.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
